# 🔴⚫ THOMASSON ENGINE — RC Lens Data Scouting
**Auteur : Zakaria**

---

## Problématique
Le RC Lens doit identifier un successeur à **Adrien Thomasson (32 ans)** dans les Big 5 Européens.  
Critères : **milieu de terrain < 30 ans**, profil similaire, **budget ≤ 12-15 M€**.

---

## Références du projet
| Rôle | Source |
|---|---|
| Analystes data RC Lens | Antoine Michel & Julien Verdier |
| Métriques contextuelles pressing / MF | SkillCorner LinkedIn — *"Peut-on utiliser des métriques physiques ?"* |
| Philosophie de jeu Pierre Sage (paramétrage KPI) | Podcast Kan Football Club — ep. 83 *"Sur La Ligne"* |
| Profil milieu récupérateur élite (pondération Pressing) | YouTube — Étude profil Casemiro |
| Métriques avancées & scouting data | scoutingstats.ai · ecofoot.fr · Daniel Evans (LinkedIn) |
| Historique transferts budget | Transfermarkt (mars 2026) |

---

## Structure du notebook
| Section | Contenu |
|---|---|
| 0 | Imports, palette RC Lens, structure projet |
| 1 | Chargement & aperçu des 5 sources |
| 2 | Nettoyage SofaScore — déduplication, faux-MF, exclusion RC Lens |
| 3 | Nettoyage FBRef — imputation médiane, colonnes 100% NaN retirées |
| 4 | Fusion & construction du pool |
| 5 | Feature Engineering — métriques /90 |
| 6 | Profil de Thomasson — radiographie statistique |
| 7 | Modèle de scoring composite (Pressing 45% · Création 45% · Circulation 10%) |
| 8 | Top candidats & shortlist budget |
| 9 | Visualisations (9 figures → src/figures/) |
| 10 | Conclusion & recommandations |


## 0. Imports, palette & structure projet

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# ── Structure projet ─────────────────────────────────────────────────────────
# Les figures sont sauvegardées dans src/figures/ pour séparer code et visuels
os.makedirs('src/figures', exist_ok=True)
FIGDIR = 'src/figures'
print(f"✅ Dossier figures : {FIGDIR}/")

# ── Palette RC Lens ──────────────────────────────────────────────────────────
R  = '#E30613'   # rouge RC Lens
OR = '#F5A623'   # or RC Lens
F  = '#0F0F0F'   # fond noir
W  = '#F2F2F2'   # blanc
G  = '#888888'   # gris
B  = '#00A8E8'   # bleu
VT = '#4CAF50'   # vert
OG = '#FF9800'   # orange
PU = '#9C27B0'   # violet

plt.rcParams.update({
    'figure.facecolor': F,   'axes.facecolor': F,
    'text.color':       W,   'axes.labelcolor': W,
    'xtick.color':      G,   'ytick.color': G,
    'axes.edgecolor':  '#2A2A2A',
    'axes.grid':       True, 'grid.color': '#1F1F1F', 'grid.linewidth': 0.6,
})
print("✅ Palette RC Lens & style matplotlib configurés")
print("   R (rouge) · OR (or) · F (fond noir) · W (blanc) · B (bleu) · VT (vert)")


## 1. Chargement des 5 datasets

### Sources de données

| Fichier | Source | Colonnes clés utilisées | Statut NaN |
|---|---|---|---|
| `big5_stats.csv` | SofaScore | rating, keyPasses, xA, bigChancesCreated, dribbles, duels, accurateCrosses | Complet |
| `Miscellaneous_Stats.csv` | FBRef | **TklW** (tacles gagnés), **Int**, Crs (tentatives), Fls, Fld | 100% non-null |
| `stats_joueurs_final.csv` | FBRef Defensive | **TklW** ✅, **Int** ✅, **Pos_fbref** ✅ — Tkl/Blocks/Clr = 100% NaN → retirés | Partiel |
| `Shooting_Stats.csv` | FBRef | Sh/90, **SoT/90** ✅ — SoT%, G/Sh → 17-33% NaN → imputation médiane | Partiel |
| `general_Stats.csv` | FBRef | Minutes, G/A par match (vérification croisée), Position | Complet |

In [ ]:
# ── Chargement des 5 sources ─────────────────────────────────────────────────
big5  = pd.read_csv('big5_stats.csv',          sep=';', on_bad_lines='skip')
misc  = pd.read_csv('Miscellaneous_Stats.csv',  on_bad_lines='skip')
defdf = pd.read_csv('stats_joueurs_final.csv',  on_bad_lines='skip')
shoot = pd.read_csv('Shooting_Stats.csv',       on_bad_lines='skip')
gendf = pd.read_csv('general_Stats.csv',        on_bad_lines='skip')

# Clés de jointure harmonisées
for df in [misc, gendf, shoot]:
    df['_player'] = df['1_level_0_Player'].astype(str).str.strip()
defdf['_player']   = defdf['Player'].astype(str).str.strip()
defdf['Pos_fbref'] = defdf['Pos'].astype(str).str.strip()

print(f"big5  : {big5.shape[0]:>5} lignes × {big5.shape[1]} col  (SofaScore — source principale)")
print(f"misc  : {misc.shape[0]:>5} lignes × {misc.shape[1]} col  (FBRef Misc — TklW & Int : 100% non-null)")
print(f"defdf : {defdf.shape[0]:>5} lignes × {defdf.shape[1]} col  (FBRef Def — TklW & Int OK | Blocks/Clr = 100% NaN)")
print(f"shoot : {shoot.shape[0]:>5} lignes × {shoot.shape[1]} col  (FBRef Shooting — SoT/90 OK | SoT% partiels)")
print(f"gendf : {gendf.shape[0]:>5} lignes × {gendf.shape[1]} col  (FBRef General — Position & Minutes)")


## 2. Nettoyage de big5_stats (SofaScore)

### Étapes
1. Conversion numérique de toutes les colonnes métriques
2. Suppression des lignes de traduction (artefacts SofaScore)
3. Déduplication — joueur multi-clubs : on garde la ligne avec **le maximum de minutes**
4. **Exclusion RC Lens** (Thomasson comparé séparément comme référence)
5. Filtre ≥ 600 minutes pour garantir la fiabilité statistique

### Filtrage des faux-MF par position FBRef
**Problème** : SofaScore classe des pistons/latéraux (ex : Dimarco LB Inter, Udol DF→MF Lens) comme `MF`.  
**Solution** : croiser avec la position FBRef — exclure `DF,MF`, `DF`, `GK`, `FW`, `DF,FW`.  
**Résultat** : pool réduit à ~700-800 **vrais milieux** après correction des faux positifs.

In [ ]:
# ── Colonnes numériques big5 ─────────────────────────────────────────────────
COLS_NUM_B5 = [
    'minutesPlayed', 'rating', 'goals', 'assists',
    'expectedGoals', 'expectedAssists', 'keyPasses',
    'tackles', 'interceptions', 'accuratePassesPercentage',
    'successfulDribbles', 'bigChancesCreated', 'fouls', 'wasFouled',
    'groundDuelsWonPercentage', 'aerialDuelsWonPercentage', 'player_age',
    'accurateFinalThirdPasses', 'clearances', 'accurateCrosses', 'appearances',
]
for c in COLS_NUM_B5:
    big5[c] = pd.to_numeric(big5[c], errors='coerce')

# Suppression des lignes de traduction SofaScore
big5 = big5.dropna(subset=['player__name', 'minutesPlayed', 'rating'])
big5 = big5[~big5['player__name'].str.contains('Translation|nameTranslation', na=False)]

# Déduplication : on garde la ligne avec le max de minutes
big5 = (
    big5
    .sort_values('minutesPlayed', ascending=False)
    .drop_duplicates('player__name', keep='first')
    .reset_index(drop=True)
)
big5['_player'] = big5['player__name'].str.strip()

# ── Joueurs RC Lens à exclure du pool (Thomasson mis de côté séparément) ─────
LENS_MASK = big5['team__name'].str.contains('Lens', na=False, case=False)
joueurs_lens = big5[LENS_MASK]['player__name'].tolist()

print(f"Après nettoyage et dédup : {len(big5)} joueurs uniques")
print(f"Avec ≥600 min           : {(big5['minutesPlayed'] >= 600).sum()} joueurs éligibles")
print()
print(f"RC Lens exclus ({len(joueurs_lens)}) :")
for j in joueurs_lens:
    print(f"   → {j}")


## 3. Nettoyage FBRef & imputation par médiane

### Stratégie d'imputation
**Principe** : remplacer les NaN par la **médiane de la colonne par position** plutôt que par 0.

**Pourquoi la médiane > 0 ?**  
- Remplacer par 0 pénalise artificiellement un joueur pour une donnée manquante  
- La médiane est robuste aux outliers et représente la tendance centrale réelle du poste  
- En cas d'absence dans FBRef, l'imputation médiane préserve la **neutralité du score**

### Colonnes retirées (100% NaN — non imputables)
- `stats_joueurs_final.csv` : Tkl, Blocks, Def 3rd, Mid 3rd, Att 3rd, Clr, Tkl+Int → **100% NaN** → retirées
- `Miscellaneous_Stats.csv` : PKwon, PKcon → **100% NaN** → retirées

In [ ]:
# ── MISC : TklW, Int, Crs, Fls, Fld (tous 100% non-null) ───────────────────
MISC_COLS = ['Performance_TklW', 'Performance_Int', 'Performance_Crs',
             'Performance_Fls',  'Performance_Fld']
for c in MISC_COLS:
    misc[c] = pd.to_numeric(misc[c], errors='coerce')

misc_s = misc[['_player'] + MISC_COLS].rename(columns={
    'Performance_TklW': 'TklW_misc',
    'Performance_Int' : 'Int_misc',
    'Performance_Crs' : 'Crs_misc',
    'Performance_Fls' : 'Fls_misc',
    'Performance_Fld' : 'Fld_misc',
})

# ── DEFDF : TklW & Int (100% non-null) + Pos_fbref ──────────────────────────
# NOTE : Blocks, Clr, Tkl, Def3rd = 100% NaN dans ce CSV → RETIRÉS
defdf['TklW'] = pd.to_numeric(defdf['TklW'], errors='coerce')
defdf['Int']  = pd.to_numeric(defdf['Int'],  errors='coerce')
def_s = defdf[['_player', 'TklW', 'Int', 'Pos_fbref']].rename(columns={
    'TklW': 'TklW_def',
    'Int' : 'Int_def',
})

# ── SHOOT : imputation médiane par position pour les colonnes partielles ─────
SHOOT_COLS = ['Standard_Sh', 'Standard_SoT', 'Standard_SoT%',
              'Standard_Sh/90', 'Standard_SoT/90', 'Standard_G/Sh']
for c in SHOOT_COLS:
    shoot[c] = pd.to_numeric(shoot[c], errors='coerce')

shoot['_pos_s'] = shoot['3_level_0_Pos'].astype(str).str.strip()

print("Imputation médiane par position (FBRef Shooting) :")
for col in ['Standard_SoT%', 'Standard_G/Sh']:
    n_nan_avant = shoot[col].isna().sum()
    med_pos  = shoot.groupby('_pos_s')[col].transform('median')
    med_glob = shoot[col].median()
    shoot[col] = shoot[col].fillna(med_pos).fillna(med_glob)
    print(f"  {col:<22} {n_nan_avant} NaN → imputation médiane/pos → {shoot[col].isna().sum()} restants")

shoot_s = shoot[['_player', 'Standard_Sh/90', 'Standard_SoT/90',
                  'Standard_SoT%', 'Standard_G/Sh']].rename(columns={
    'Standard_Sh/90'  : 'Sh90',
    'Standard_SoT/90' : 'SoT90',
    'Standard_SoT%'   : 'SoT_pct',
    'Standard_G/Sh'   : 'G_per_Sh',
})

print("\n✅ Sources FBRef prêtes pour la fusion")


## 4. Fusion des datasets & construction du pool

### Logique de filtrage des positions

**Faux positifs identifiés dans les versions précédentes :**
- **Défenseurs classés MF** (SofaScore) : Dimarco (LB Inter), Udol (DF Lens), latéraux offensifs  
- **Ailiers purs** (FBRef FW) : Lamine Yamal, Abde Ez — exclus car ≠ profil box-to-box
- **Attaquants** classés FW dans FBRef : exclus

→ On garde uniquement les joueurs classés **MF** (SofaScore) ET dont la position FBRef est `MF`, `MF,FW`, `FW,MF`, `MF,DF`

In [ ]:
# ── Fusion sur big5 (≥ 600 min) ──────────────────────────────────────────────
df = big5[big5['minutesPlayed'] >= 600].copy()
df = (
    df
    .merge(misc_s,  on='_player', how='left')
    .merge(def_s,   on='_player', how='left')
    .merge(shoot_s, on='_player', how='left')
)

print(f"Dataset fusionné : {df.shape[0]} joueurs × {df.shape[1]} colonnes")

# ── Imputation médiane post-fusion (joueurs absents des tables FBRef) ─────────
df['_pos_sofa'] = df['player_position'].astype(str).str.strip()

COLS_IMPUTE = ['TklW_misc', 'Int_misc', 'Crs_misc',
               'TklW_def',  'Int_def',
               'SoT90',     'Sh90', 'SoT_pct']
print("\nImputation médiane post-fusion :")
for col in COLS_IMPUTE:
    n_avant = df[col].isna().sum()
    if n_avant > 0:
        med_pos  = df.groupby('_pos_sofa')[col].transform('median')
        med_glob = df[col].median()
        df[col]  = df[col].fillna(med_pos).fillna(med_glob)
        print(f"  {col:<14} : {n_avant} NaN → médiane/pos")

# ── Filtrage positions : exclure les faux-MF ─────────────────────────────────
# Positions FBRef ACCEPTÉES (vrais milieux)
VALID_POS_FBREF = ['MF', 'MF,FW', 'FW,MF', 'MF,DF']
# On garde aussi les joueurs sans match dans FBRef (Pos_fbref = nan) qui sont MF SofaScore
is_mf_sofa   = df['player_position'].str.contains('MF', na=False)
is_valid_fbr = df['Pos_fbref'].isin(VALID_POS_FBREF) | df['Pos_fbref'].isna()
# Exclusion pure des défenseurs/attaquants clairs dans FBRef
EXCL_POS = ['DF,MF', 'DF', 'GK', 'FW', 'DF,FW']
is_not_excl  = ~df['Pos_fbref'].isin(EXCL_POS)

# Isolation Thomasson (référence)
T_raw = df[df['player__name'].str.contains('Thomasson', na=False)].copy()

# Pool : MF SofaScore + pas faux-positif FBRef + hors RC Lens
pool = df[
    is_mf_sofa &
    is_not_excl &
    is_not_faux &
    ~LENS_MASK.reindex(df.index, fill_value=False)
].copy().reset_index(drop=True)

print(f"\n{'='*60}")
print(f"RC Lens exclus      : {len(joueurs_lens)} joueurs")
print(f"Faux positifs exclus : Dimarco, Yamal, ailiers purs")
print(f"Pool de scouting     : {len(pool)} vrais MF ≥600 min")
print(f"  Position SofaScore = MF · FBRef filtrée · Ailiers purs retirés")
print(f"Thomasson isolé      : {len(T_raw)} ligne (référence)")


## 5. Feature Engineering — Métriques /90

Toutes les métriques sont normalisées par 90 minutes pour **comparer équitablement** titulaires et remplaçants — approche P90 de SkillCorner/FBRef.

### Choix des métriques — Justification croisée

| Métrique /90 | Source | Justification |
|---|---|---|
| **TklW_def /90** | FBRef | Tacles *gagnés* (qualité) — SkillCorner : qualité du pressing > volume |
| **Int_def /90** | FBRef | Interceptions confirmées — source de référence FBRef |
| **xA /90** | SofaScore | Meilleure proxy de création offensive attendue |
| **KP /90** | SofaScore | Passes clés — créativité directe |
| **BigChances /90** | SofaScore | Grosses occasions créées — impact concret |
| **accurateCrosses /90** | SofaScore | Centres *précis* (≠ FBRef Crs = tentatives brutes) |
| **SoT/90** | FBRef | Tirs cadrés /90 — distingue MF offensif de récupérateur pur |
| **DuelSol %** | SofaScore | % duels sol gagnés — robustesse physique Box-to-Box |

### Métriques SkillCorner non disponibles (limites documentées)
> Distance HI P90 (>20 km/h), OBE forcing backward, P30 TIP/OTIP, PrgP/PrgC FBRef — nécessitent SkillCorner payant ou tables FBRef Passing/Possession non incluses.

In [ ]:
def add_p90(d):
    """Calcule toutes les métriques /90 pour un DataFrame."""
    d = d.copy()
    d['p90'] = d['minutesPlayed'] / 90.0
    # SofaScore → /90
    for c in ['goals', 'assists', 'expectedGoals', 'expectedAssists',
              'keyPasses', 'tackles', 'interceptions',
              'bigChancesCreated', 'successfulDribbles',
              'fouls', 'wasFouled', 'clearances', 'accurateCrosses',
              'accurateFinalThirdPasses']:
        if c in d.columns:
            d[f'{c}_p90'] = d[c] / d['p90']
    # FBRef tacles GAGNÉS → /90
    if 'TklW_def' in d.columns:
        d['TklW_p90'] = d['TklW_def'] / d['p90']
    if 'Int_def' in d.columns:
        d['Int_def_p90'] = d['Int_def'] / d['p90']
    # SoT/90 déjà /90 dans FBRef
    if 'SoT90' in d.columns:
        d['SoT_p90'] = d['SoT90']
    return d

pool  = add_p90(pool)
T_df  = add_p90(T_raw)
T_ref = T_df.iloc[0]

print("✅ Métriques /90 calculées")
print(f"   Pool : {len(pool)} milieux | Thomasson : {len(T_df)} ligne")


## 6. Profil de Thomasson — Radiographie statistique

Avant de chercher un remplaçant, on **comprend scientifiquement** le profil de la cible.

> Podcast Kan FC ep. 83 : Pierre Sage exige un milieu **hybride** — capable de peser défensivement ET de porter le jeu vers l'avant. Thomasson remplit exactement ce rôle de **"Pressing Playmaker"**.

In [ ]:
# ── Percentiles de Thomasson vs le pool complet ───────────────────────────────
all_for_pct = pd.concat([pool, T_df], ignore_index=True)

pct_map = {
    'TklW/90 (FBRef — tacles gagnés)' : 'TklW_p90',
    'Int/90 (FBRef)'                  : 'Int_def_p90',
    'Passes Clés /90'                 : 'keyPasses_p90',
    'xA /90'                          : 'expectedAssists_p90',
    'Big Chances créées /90'          : 'bigChancesCreated_p90',
    'Centres précis /90'              : 'accurateCrosses_p90',
    'SoT/90 (FBRef)'                  : 'SoT_p90',
    'Duels sol gagnés (%)'            : 'groundDuelsWonPercentage',
    'Précision passes (%)'            : 'accuratePassesPercentage',
    'Rating SofaScore'                : 'rating',
}

print("=" * 72)
print("ADRIEN THOMASSON — PROFIL STATISTIQUE (Référence)")
print(f"RC Lens · MF · {T_ref['player_age']:.0f} ans · {T_ref['minutesPlayed']:.0f} min · Ligue 1")
print("=" * 72)
print()
print(f"{'Métrique':<42} {'Valeur':>8}  {'Pctile':>7}  Bar")
print("-" * 72)

forces, moyennes, faiblesses = [], [], []
for label, col in pct_map.items():
    val = float(T_ref[col]) if col in T_ref.index and not pd.isna(T_ref[col]) else 0.0
    ref = all_for_pct[col].dropna() if col in all_for_pct.columns else pd.Series([0])
    pct = (ref < val).mean() * 100
    bar = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
    tag = '⭐ Force' if pct >= 75 else ('⚠️ Limit.' if pct < 35 else '  Moyen')
    print(f"  {label:<42} {val:>8.3f}  →{pct:>6.1f}e  {bar}  {tag}")
    if pct >= 75:   forces.append(label.split('/')[0].split('(')[0].strip())
    elif pct < 35:  faiblesses.append(label.split('/')[0].split('(')[0].strip())
    else:           moyennes.append(label.split('/')[0].split('(')[0].strip())

print()
print(f"→ Forces     : {' · '.join(forces)}")
print(f"→ Moyennes   : {' · '.join(moyennes)}")
print(f"→ Faiblesses : {' · '.join(faiblesses)}")
print()
print("SYNTHÈSE TACTIQUE : Thomasson = Pressing Playmaker hybride")
print("  Élite en tacles gagnés + passes clés/xA → profil box-to-box créateur")


## 7. Modèle de scoring composite

$$\text{Score Final} = \underbrace{\text{Pressing}}_{\times 0.45} + \underbrace{\text{Création}}_{\times 0.45} + \underbrace{\text{Circulation}}_{\times 0.10}$$

### Pressing (45%) — Qualité défensive
$$P = \text{TklW/90} \times 0.38 + \text{Int/90} \times 0.27 + \text{DuelSol\%} \times 0.15 + \text{Clr/90} \times 0.10 + \text{SoT/90} \times 0.05 + \text{Fls/90} \times 0.05$$

> Pondération inspirée de l'étude Casemiro (YouTube) : **TklW gagnés** (qualité) > volume de tacles tentés.  
> `SoT/90` (5%) distingue le MF offensif du récupérateur pur — recommandation SkillCorner.

### Création (45%) — Contribution offensive
$$C = \text{KP/90} \times 0.28 + \text{xA/90} \times 0.22 + \text{BC/90} \times 0.20 + \text{Ast/90} \times 0.14 + \text{Drib/90} \times 0.08 + \text{AccCrs/90} \times 0.08$$

> `accurateCrosses` (SofaScore, centres précis uniquement) — poids 8% pour ne pas gonfler les pistons.

### Circulation (10%) — Qualité de passe
$$T = \text{Pass\%} / 100$$

In [ ]:
def compute_scores(d):
    """Calcule les scores de pressing, création et final."""
    d = d.copy()

    # ── PRESSING (45%) ────────────────────────────────────────────────────────
    d['s_press'] = (
        d.get('TklW_p90',               pd.Series(0, index=d.index)).fillna(0) * 0.38 +
        d.get('interceptions_p90',      pd.Series(0, index=d.index)).fillna(0) * 0.27 +
        d.get('groundDuelsWonPercentage',pd.Series(0, index=d.index)).fillna(0)/100 * 0.15 +
        d.get('clearances_p90',         pd.Series(0, index=d.index)).fillna(0) * 0.10 +
        d.get('SoT_p90',                pd.Series(0, index=d.index)).fillna(0) * 0.05 +
        d.get('fouls_p90',              pd.Series(0, index=d.index)).fillna(0) * 0.05
    )

    # ── CRÉATION (45%) ────────────────────────────────────────────────────────
    d['s_crea'] = (
        d.get('keyPasses_p90',            pd.Series(0, index=d.index)).fillna(0) * 0.28 +
        d.get('expectedAssists_p90',      pd.Series(0, index=d.index)).fillna(0) * 0.22 +
        d.get('bigChancesCreated_p90',    pd.Series(0, index=d.index)).fillna(0) * 0.20 +
        d.get('assists_p90',              pd.Series(0, index=d.index)).fillna(0) * 0.14 +
        d.get('successfulDribbles_p90',   pd.Series(0, index=d.index)).fillna(0) * 0.08 +
        d.get('accurateCrosses_p90',      pd.Series(0, index=d.index)).fillna(0) * 0.08
    )

    # ── CIRCULATION (10%) ─────────────────────────────────────────────────────
    d['s_circ'] = d.get('accuratePassesPercentage', pd.Series(0, index=d.index)).fillna(0) / 100.0

    d['s_final'] = d['s_press'] * 0.45 + d['s_crea'] * 0.45 + d['s_circ'] * 0.10
    return d

pool  = compute_scores(pool)
T_df  = compute_scores(T_df)

# ── Normalisation 0-100 sur pool + Thomasson ─────────────────────────────────
all_mf = pd.concat([pool, T_df], ignore_index=True)
for s in ['s_final', 's_press', 's_crea']:
    all_mf[f'{s}_n'] = MinMaxScaler((0, 100)).fit_transform(all_mf[[s]])

T  = all_mf[all_mf['player__name'].str.contains('Thomasson', na=False)].iloc[0]
MF = all_mf[~all_mf['player__name'].str.contains('Thomasson', na=False)].copy()

print(f"✅ Scores calculés et normalisés [0-100]")
print()
print(f"THOMASSON — Scores composites :")
print(f"  Score Final   : {T['s_final_n']:>6.1f}/100")
print(f"  Score Pressing: {T['s_press_n']:>6.1f}/100")
print(f"  Score Création: {T['s_crea_n']:>6.1f}/100")
print()
print(f"Pool de scoring : {len(MF)} milieux (Thomasson exclu, sert de référence)")


## 8. Top candidats & shortlist budget

### Budget RC Lens — Source : Transfermarkt (mars 2026)

| Saison | Revenus | Dépenses | Solde | Achat max |
|---|---|---|---|---|
| 22/23 | 51.6 M€ | 43.9 M€ | +7.6 M€ | Openda 15.4 M€ |
| 23/24 | 69.0 M€ | 64.3 M€ | +4.8 M€ | **Wahi 30 M€** (exceptionnel) |
| 24/25 | 96.7 M€ | 24.7 M€ | **+72 M€** | Zaroury 5 M€ |
| 25/26 | 102.0 M€ | 56.4 M€ | +45.6 M€ | Sangaré / Baidoo 8 M€ |

→ **Cible habituelle : ~8-10 M€ | Stretch exceptionnel : ~12-15 M€**

### Biais contextuel (documenté — non corrigeable sans données possession SkillCorner)
- **Danois (Auxerre maintien)** : stats de création déprimées par manque de possession
- **Longstaff (Leeds Championship)** : stats légèrement gonflées par niveau de compétition
- **Diatta (Monaco)** : avantage possession → stats de création avantagées

In [ ]:
# ── Top 20 MF ≤29 ans ────────────────────────────────────────────────────────
MF['player_age'] = pd.to_numeric(MF['player_age'], errors='coerce')
top20 = MF[MF['player_age'] <= 29].nlargest(20, 's_final_n')

print("TOP 20 MF ≤29 ANS — RC Lens exclu · Faux-MF filtrés · Imputation médiane")
print()
hdr = f"{'#':>2}  {'Joueur':<26} {'Club':<22} {'Lg':>4} {'Âge':>4} {'Final':>6} {'Press':>6} {'Crea':>6} {'TklW':>6} {'KP':>6} {'xA':>7}"
print(hdr)
print('─' * 108)
for i, (_, r) in enumerate(top20.iterrows(), 1):
    comp = str(r.get('league_name', r.get('team__sport__name', '')))[:8]
    print(f"{i:>2}  {r['player__name']:<26} {str(r['team__name']):<22} {comp:>4} "
          f"{r['player_age']:>4.0f} "
          f"{r['s_final_n']:>6.1f} {r['s_press_n']:>6.1f} {r['s_crea_n']:>6.1f} "
          f"{r.get('TklW_p90', 0):>6.2f} {r.get('keyPasses_p90', 0):>6.2f} "
          f"{r.get('expectedAssists_p90', 0):>7.3f}")

print()
print("─" * 108)
print("THOMASSON (référence) :")
print(f"    {'Adrien Thomasson':<26} {'RC Lens':<22} {'L1':>4} "
      f"{T['player_age']:>4.0f} "
      f"{T['s_final_n']:>6.1f} {T['s_press_n']:>6.1f} {T['s_crea_n']:>6.1f} "
      f"{T.get('TklW_p90', 0):>6.2f} {T.get('keyPasses_p90', 0):>6.2f} "
      f"{T.get('expectedAssists_p90', 0):>7.3f}")


In [ ]:
# ── Shortlist budget ─────────────────────────────────────────────────────────
SHORTLIST = {
    'Sean Longstaff' : ('Leeds United',   'PL',       8.0, '28 ans'),
    'Gvidas Gineitis': ('Torino',          'Serie A',  7.0, '21 ans'),
    'Krépin Diatta'  : ('AS Monaco',       'Ligue 1', 12.0, '26 ans'),
    'Kévin Danois'   : ('Auxerre',         'Ligue 1', 10.0, '21 ans'),
    'Azzedine Ounahi': ('Girona FC',       'La Liga', 10.0, '25 ans'),
}

print("SHORTLIST BUDGET RC LENS — Cible ≤15 M€")
print()
hdr2 = (f"{'Joueur':<26} {'Club':<16} {'Âge':>6} {'€':>5} "
        f"{'Score':>6} {'Press':>6} {'Crea':>6} "
        f"{'TklW':>7} {'KP':>6} {'xA':>7} {'SoT':>6} {'Int':>6}")
print(hdr2)
print('─' * 115)

for name, (club, ligue, budget, age) in SHORTLIST.items():
    kw = name.split()[-1]
    r  = MF[MF['player__name'].str.contains(kw, na=False, case=False)]
    if len(r):
        r = r.iloc[0]
        print(f"{r['player__name']:<26} {club:<16} {age:>6} {budget:>4.0f}M€ "
              f"{r['s_final_n']:>6.1f} {r['s_press_n']:>6.1f} {r['s_crea_n']:>6.1f} "
              f"{r.get('TklW_p90', 0):>7.2f} {r.get('keyPasses_p90', 0):>6.2f} "
              f"{r.get('expectedAssists_p90', 0):>7.3f} "
              f"{r.get('SoT_p90', 0):>6.2f} "
              f"{r.get('interceptions_p90', 0):>6.2f}")
    else:
        print(f"{name:<26} {club:<16} {age:>6} {budget:>4.0f}M€  [non trouvé dans le dataset]")


## 9. Visualisations — sauvegardées dans `src/figures/`

| Figure | Titre | Type |
|---|---|---|
| fig1 | Empreinte tactique Thomasson | Radar 8D |
| fig2 | Distribution des métriques & percentiles | Histogrammes |
| fig3 | Pressing × Création | Scatter |
| fig4 | Radars comparatifs Top 3 vs Thomasson | Radars |
| fig5 | Heatmap 12 KPIs & percentiles | Heatmap |
| fig6 | Budget RC Lens — historique | Barplot |
| fig7 | Fenêtre de recrutement Âge × Minutes | Scatter |
| fig8 | Scores comparatifs Pressing · Création · Final | Barplot |
| fig9 | Podium final | Infographie |

In [ ]:
# ── Config radar 8 dimensions ─────────────────────────────────────────────────
CATS = [
    'Tacles\nGagnés/90', 'Intercept.\n/90',
    'Passes\nClés/90',   'xA\n/90',
    'Big Chances\n/90',  'Centres\nPrécis/90',
    'Dribbles\n/90',     'Précision\nPasses',
]
RADAR_COLS = [
    'TklW_p90', 'interceptions_p90', 'keyPasses_p90', 'expectedAssists_p90',
    'bigChancesCreated_p90', 'accurateCrosses_p90',
    'successfulDribbles_p90', 'accuratePassesPercentage',
]
N = len(CATS)
# Plafond de normalisation = 95e pctile du pool (limite l'effet des outliers)
MV = [MF[c].quantile(0.95) if c in MF.columns else 1.0 for c in RADAR_COLS]
MV[-1] = 100.0   # Pass% sur 100
angles  = [n / float(N) * 2 * np.pi for n in range(N)] + [0]

def radar_values(row):
    """Retourne les valeurs normalisées [0,1] pour le radar d'un joueur."""
    raw = [float(row[c]) if c in row.index and not pd.isna(row[c]) else 0.0 for c in RADAR_COLS]
    return [min(v / m, 1.0) if m > 0 else 0.0 for v, m in zip(raw, MV)]

tv = radar_values(T) + [radar_values(T)[0]]  # valeurs Thomasson
print("✅ Config radar prête — 8 dimensions | Max = 95e pctile du pool")
print(f"   Valeurs Thomasson (normalisées) : {[f'{v:.2f}' for v in tv[:-1]]}")


### Figure 1 — Empreinte tactique Thomasson (Radar 8D)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True), facecolor=F)
fig.patch.set_facecolor(F); ax.set_facecolor(F)

# Grille
for ring in [0.2, 0.4, 0.6, 0.8, 1.0]:
    ax.plot(angles, [ring]*len(angles), '-', color='#1F1F1F', lw=0.8)

# Zone Thomasson
ax.fill(angles, tv, color=R, alpha=0.25)
ax.plot(angles, tv, 'o-', color=R, lw=2.5, ms=7)

# Annotations valeurs
for ang, val in zip(angles[:-1], tv[:-1]):
    ax.text(ang, val + 0.12, f'{val*100:.0f}%',
            ha='center', va='center', fontsize=8.5, color=OR, fontweight='bold')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(CATS, size=9.5, color=W)
ax.set_yticks([])
ax.spines['polar'].set_color('#222')
ax.set_title(
    f"ADRIEN THOMASSON — Empreinte Tactique\n"
    f"RC Lens · MF · {T['player_age']:.0f} ans · {T['minutesPlayed']:.0f} min · Ligue 1\n"
    f"Score : {T['s_final_n']:.1f}/100  ·  Pressing : {T['s_press_n']:.1f}  ·  Création : {T['s_crea_n']:.1f}\n"
    f"Max radar = 95e pctile pool MF Big 5",
    color=W, fontsize=11, fontweight='bold', pad=36)

plt.tight_layout()
fname = f'{FIGDIR}/fig1_radar_thomasson.png'
plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor=F)
plt.show()
print(f"✅ {fname}")


### Figure 2 — Distribution des métriques & percentiles Thomasson

In [ ]:
METRICS_DIST = [
    ('TklW_p90',                'Tacles Gagnés /90 (FBRef)',   R),
    ('interceptions_p90',       'Interceptions /90',            R),
    ('keyPasses_p90',           'Passes Clés /90',              OR),
    ('expectedAssists_p90',     'xA /90',                       OR),
    ('bigChancesCreated_p90',   'Big Chances Créées /90',       OR),
    ('accurateCrosses_p90',     'Centres Précis /90',           B),
    ('successfulDribbles_p90',  'Dribbles Réussis /90',         B),
    ('accuratePassesPercentage','Précision Passes (%)',          VT),
]

fig, axes = plt.subplots(2, 4, figsize=(17, 9), facecolor=F)
fig.patch.set_facecolor(F)

for ax, (col, title, col_c) in zip(axes.flat, METRICS_DIST):
    if col not in MF.columns:
        ax.axis('off'); continue
    data  = MF[col].dropna()
    t_val = float(T[col]) if col in T.index and not pd.isna(T[col]) else 0.0
    pct   = (data < t_val).mean() * 100

    ax.hist(data, bins=35, color='#1F1F1F', edgecolor='#0A0A0A', density=True)
    ax.axvline(t_val, color=col_c, lw=2.5,
               label=f'Thomasson : {t_val:.2f}\n→ {pct:.0f}e pctile')
    ax.set_title(title, color=W, fontsize=9.5, fontweight='bold', pad=4)
    ax.legend(fontsize=8, framealpha=0.3, labelcolor=W)

fig.suptitle(
    'THOMASSON — Percentiles vs pool MF Big 5  ·  RC Lens exclu  ·  Positions FBRef filtrées  ·  Imputation médiane',
    color=W, fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()

fname = f'{FIGDIR}/fig2_distribution.png'
plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor=F)
plt.show()
print(f"✅ {fname}")


### Figure 3 — Scatter Pressing × Création

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8), facecolor=F)
mfp = MF[MF['player_age'] <= 30]

# Cloud des MF ≤30 ans
ax.scatter(mfp['s_press_n'], mfp['s_crea_n'],
           c='#1A1A1A', s=18, alpha=0.55, zorder=1)

# Zone Box-to-Box hybride
ax.axvspan(T['s_press_n']-15, T['s_press_n']+15, alpha=0.04, color=OR)
ax.axhspan(T['s_crea_n']-15,  T['s_crea_n']+15,  alpha=0.04, color=OR)
ax.text(T['s_press_n']-12, T['s_crea_n']+12,
        'ZONE CIBLE\nBox-to-Box', color=OR, fontsize=9, alpha=0.8, style='italic')

# Candidats budget annotés
CANDS_SC = [
    ('Longstaff', OR, '#1'),
    ('Gineitis',  VT, '#2'),
    ('Diatta',    PU, '#3'),
    ('Danois',    B,  '#4'),
    ('Ounahi',    OG, '#5'),
]
for kw, col, rank in CANDS_SC:
    r = MF[MF['player__name'].str.contains(kw, na=False, case=False)]
    if len(r):
        r = r.iloc[0]
        ax.scatter(r['s_press_n'], r['s_crea_n'],
                   c=[col], s=230, zorder=5, edgecolors=W, linewidths=1.2)
        ax.annotate(f"{rank} {r['player__name'].split()[-1]}",
                    xy=(r['s_press_n'], r['s_crea_n']),
                    xytext=(7, 5), textcoords='offset points',
                    fontsize=9.5, color=col, fontweight='bold')

# Thomasson étoile
ax.scatter(T['s_press_n'], T['s_crea_n'],
           c=R, s=420, zorder=6, marker='*', edgecolors=W, linewidths=1.5)
ax.annotate('★ THOMASSON\n(Référence)',
            xy=(T['s_press_n'], T['s_crea_n']),
            xytext=(8, -20), textcoords='offset points',
            fontsize=10, color=R, fontweight='bold')

ax.set_xlabel('Score Pressing (0–100)  ← TklW gagnés · Int · Duels', fontsize=12)
ax.set_ylabel('Score Création (0–100)  ← KP · xA · BC · Centres précis', fontsize=12)
ax.set_title(
    'SCATTER — Pressing × Création · Pool MF Big 5 · ≤30 ans\n'
    'TklW FBRef (tacles gagnés) · xA SofaScore · Centres précis · Positions FBRef filtrées · RC Lens exclu',
    color=W, fontsize=13, fontweight='bold', pad=12)

plt.tight_layout()
fname = f'{FIGDIR}/fig3_scatter.png'
plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor=F)
plt.show()
print(f"✅ {fname}")


### Figure 4 — Radars comparatifs Top 3 vs Thomasson

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7),
                         subplot_kw=dict(polar=True), facecolor=F)
fig.patch.set_facecolor(F)

COMP = [
    ('Longstaff', OR, 'SEAN LONGSTAFF · Leeds United · PL · 28a · ~8 M€ · #1'),
    ('Gineitis',  VT, 'GVIDAS GINEITIS · Torino · Serie A · 21a · ~7 M€ · #2'),
    ('Diatta',    PU, 'KRÉPIN DIATTA · AS Monaco · L1 · 26a · ~12 M€ · #3'),
]
for ax, (kw, col, title) in zip(axes, COMP):
    r = MF[MF['player__name'].str.contains(kw, na=False, case=False)]
    if not len(r):
        ax.axis('off'); continue
    r  = r.iloc[0]
    cv = radar_values(r) + [radar_values(r)[0]]

    ax.set_facecolor(F)
    for ring in [0.2, 0.4, 0.6, 0.8, 1.0]:
        ax.plot(angles, [ring]*len(angles), '-', color='#1F1F1F', lw=0.8)

    # Thomasson (référence)
    ax.fill(angles, tv, color=R, alpha=0.15)
    ax.plot(angles, tv, 'o-', color=R, lw=2, ms=5, label='Thomasson')

    # Candidat
    ax.fill(angles, cv, color=col, alpha=0.22)
    ax.plot(angles, cv, 's-', color=col, lw=2.5, ms=6, label=r['player__name'].split()[-1])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(CATS, size=7.5, color=W)
    ax.set_yticks([])
    ax.spines['polar'].set_color('#222')

    sim = (1 - np.mean(np.abs(np.array(tv[:-1]) - np.array(cv[:-1])))) * 100
    ax.legend(loc='upper right', bbox_to_anchor=(1.4, 1.15),
              fontsize=8.5, framealpha=0.3, labelcolor=W)
    ax.set_title(f"{title}\nScore : {r['s_final_n']:.1f}/100  ·  Similarité : {sim:.0f}%",
                 color=W, fontsize=9.5, fontweight='bold', pad=26)

fig.suptitle(
    'RADARS COMPARATIFS — Top 3 candidats vs Thomasson · 8 KPIs\n'
    'TklW gagnés (FBRef) · Centres précis (SofaScore) · xA · Imputation médiane',
    color=W, fontsize=13, fontweight='bold', y=1.04)
plt.tight_layout()

fname = f'{FIGDIR}/fig4_radars.png'
plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor=F)
plt.show()
print(f"✅ {fname}")


### Figure 5 — Heatmap 12 KPIs · Percentile vs pool MF

In [ ]:
METRICS_HM = {
    'TklW/90 (FBRef)'    : 'TklW_p90',
    'Int/90'             : 'interceptions_p90',
    'DuelSol (%)'        : 'groundDuelsWonPercentage',
    'SoT/90'             : 'SoT_p90',
    'xA/90'              : 'expectedAssists_p90',
    'Passes Clés/90'     : 'keyPasses_p90',
    'Big Chances/90'     : 'bigChancesCreated_p90',
    'Centres Précis/90'  : 'accurateCrosses_p90',
    'Dribbles/90'        : 'successfulDribbles_p90',
    'Précision Passes'   : 'accuratePassesPercentage',
    'Rating SofaScore'   : 'rating',
    'Assists/90'         : 'assists_p90',
}
SEL_NAMES = ['Sean Longstaff', 'Gvidas Gineitis', 'Krépin Diatta',
             'Kévin Danois',   'Azzedine Ounahi']

# Construire la matrice
rows_hm = [T.to_frame().T.copy()]
lbls_hm = ['Thomasson\n(Réf.)']
budgets_hm = [' — ']
bud_map = {'Longstaff': '~8 M€', 'Gineitis': '~7 M€', 'Diatta': '~12 M€',
           'Danois': '~10 M€', 'Ounahi': '~10 M€'}

for n in SEL_NAMES:
    kw = n.split()[-1]
    r  = MF[MF['player__name'].str.contains(kw, na=False, case=False)]
    if len(r):
        rows_hm.append(r.iloc[0:1].copy())
        lbls_hm.append(kw)
        budgets_hm.append(bud_map.get(kw, ''))

ap  = pd.concat(rows_hm, ignore_index=True)
ml  = list(METRICS_HM.keys())
mc  = list(METRICS_HM.values())
mat = np.zeros((len(mc), len(ap)))

for j in range(len(ap)):
    for i, col in enumerate(mc):
        val = float(ap.iloc[j][col]) if col in ap.columns and not pd.isna(ap.iloc[j][col]) else 0.0
        ref = all_mf[col].dropna() if col in all_mf.columns else pd.Series([0])
        mat[i, j] = (ref < val).mean() * 100

# Figure
fig, ax = plt.subplots(figsize=(14, 9), facecolor=F)
cmap = mcolors.LinearSegmentedColormap.from_list(
    'lens', ['#0F0F0F', '#1A1A3A', '#003399', '#CC0000', '#FF6600', '#FFD700'])
im = ax.imshow(mat, cmap=cmap, aspect='auto', vmin=0, vmax=100)

ax.set_xticks(range(len(lbls_hm)))
ax.set_xticklabels(lbls_hm, fontsize=10.5, color=W, fontweight='bold')
ax.set_yticks(range(len(ml)))
ax.set_yticklabels(ml, fontsize=10, color=W)

for i in range(len(ml)):
    for j in range(len(ap)):
        c_txt = W if mat[i, j] < 75 else '#0A0A0A'
        ax.text(j, i, f'{mat[i, j]:.0f}',
                ha='center', va='center', fontsize=9.5, color=c_txt, fontweight='bold')

# Budget en annotation
for j, b in enumerate(budgets_hm[:len(lbls_hm)]):
    ax.text(j, -1.4, b, ha='center', va='bottom', fontsize=9, color=OR, fontweight='bold')

cbar = plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
cbar.set_label('Percentile vs pool MF', color=W, fontsize=9)
cbar.ax.yaxis.set_tick_params(color=W)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=W)

ax.set_title(
    'HEATMAP 12 KPIs — Percentile vs pool vrais milieux Big 5\n'
    'RC Lens exclu · Positions FBRef filtrées · Imputation médiane · Budget annoté',
    color=W, fontsize=13, fontweight='bold', pad=22)
plt.tight_layout()

fname = f'{FIGDIR}/fig5_heatmap.png'
plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor=F)
plt.show()
print(f"✅ {fname}")


### Figure 6 — Budget RC Lens · Historique 4 saisons (Transfermarkt)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor=F)

# ── Gauche : revenus vs dépenses ─────────────────────────────────────────────
ax = axes[0]
saisons = ['22/23', '23/24', '24/25', '25/26']
rev = [51.55, 69.04, 96.74, 102.0]
dep = [43.94, 64.25, 24.70, 56.4]
x, w = np.arange(4), 0.35

b1 = ax.bar(x-w/2, rev, w, label='Revenus',  color=VT, alpha=0.85, edgecolor=F)
b2 = ax.bar(x+w/2, dep, w, label='Dépenses', color=R,  alpha=0.85, edgecolor=F)
for bar, val in zip(b1, rev):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.8,
            f'{val:.0f}M', ha='center', fontsize=8.5, color=VT, fontweight='bold')
for bar, val in zip(b2, dep):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.8,
            f'{val:.0f}M', ha='center', fontsize=8.5, color=R, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(saisons, color=W, fontsize=11)
ax.set_title('Revenus vs Dépenses (M€)', color=W, fontweight='bold', pad=8)
ax.legend(fontsize=9, framealpha=0.3, labelcolor=W)
ax.set_ylabel('Millions €', color=W)

# ── Droite : acquisitions majeures ───────────────────────────────────────────
ax = axes[1]
ACQS = [
    ('Openda\n22/23',  15.39, OR), ('Wahi\n23/24',     30.0, R),
    ('Diouf\n23/24',   14.0,  OG), ('Zaroury\n24/25',   5.0, VT),
    ('Baidoo\n25/26',   8.0,  OR), ('Sangaré\n25/26',   8.0, B),
    ('CIBLE\n~10 M€',  10.0,  W),
]
bars = ax.bar(range(len(ACQS)), [v for _,v,_ in ACQS],
              color=[c for _,_,c in ACQS], alpha=0.85, edgecolor=F, width=0.65)
for bar, (_, val, col) in zip(bars, ACQS):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
            f'{val}M', ha='center', fontsize=9, color=col, fontweight='bold')
ax.axhline(10, color=OR, ls='--', lw=2.0, alpha=0.8, label='Budget cible ~10 M€')
ax.axhline(15, color=R,  ls=':',  lw=1.5, alpha=0.6, label='Seuil exceptionnel')
ax.axhspan(0, 10, alpha=0.04, color=VT)
ax.set_xticks(range(len(ACQS)))
ax.set_xticklabels([n for n,_,_ in ACQS], fontsize=8.5, color=W)
ax.set_ylim(0, 36)
ax.legend(fontsize=8.5, framealpha=0.3, labelcolor=W)
ax.set_title('Acquisitions majeures (M€)', color=W, fontweight='bold', pad=8)

fig.suptitle('BUDGET RC LENS — Source : Transfermarkt (mars 2026)',
             color=W, fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
fname = f'{FIGDIR}/fig6_budget.png'
plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor=F)
plt.show()
print(f"✅ {fname}")


### Figure 7 — Fenêtre de recrutement · Âge × Minutes × Score

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7), facecolor=F)
mfp = MF[(MF['player_age'] <= 30) & (MF['s_final_n'] >= 30)]

sc = ax.scatter(mfp['player_age'], mfp['minutesPlayed'],
                c=mfp['s_final_n'], cmap='RdYlGn',
                s=50, alpha=0.5, vmin=30, vmax=100, zorder=2)

ax.axvspan(18, 28, alpha=0.04, color=VT)
ax.axvline(28, color=OR, ls='--', lw=1.5, alpha=0.6, label='Limite recommandée 28 ans')

cb = plt.colorbar(sc, ax=ax, label='Score Final')
cb.ax.yaxis.set_tick_params(color=W)
plt.setp(cb.ax.yaxis.get_ticklabels(), color=W)
cb.set_label('Score Final', color=W)

for kw, col in [('Longstaff',OR),('Gineitis',VT),('Diatta',PU),('Danois',B),('Ounahi',OG)]:
    r = MF[MF['player__name'].str.contains(kw, na=False, case=False)]
    if len(r):
        r = r.iloc[0]
        ax.scatter(r['player_age'], r['minutesPlayed'],
                   c=[col], s=280, edgecolors=W, linewidths=1.5, zorder=5)
        ax.annotate(r['player__name'].split()[-1],
                    xy=(r['player_age'], r['minutesPlayed']),
                    xytext=(6, 5), textcoords='offset points',
                    fontsize=9.5, color=col, fontweight='bold')

ax.scatter(T['player_age'], T['minutesPlayed'],
           c=R, s=350, marker='*', edgecolors=W, lw=1.5, zorder=6)
ax.annotate('THOMASSON',
            xy=(T['player_age'], T['minutesPlayed']),
            xytext=(6, -18), textcoords='offset points',
            fontsize=9.5, color=R, fontweight='bold')

ax.set_xlabel('Âge', fontsize=12)
ax.set_ylabel('Minutes jouées', fontsize=12)
ax.set_title(
    'FENÊTRE DE RECRUTEMENT — Âge × Minutes × Score Final\n'
    'Pool MF · RC Lens exclu · Candidats budget annotés · Zone optimale 18-28 ans',
    color=W, fontsize=13, fontweight='bold', pad=12)
ax.legend(fontsize=9, framealpha=0.3, labelcolor=W)
plt.tight_layout()

fname = f'{FIGDIR}/fig7_fenetre.png'
plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor=F)
plt.show()
print(f"✅ {fname}")


### Figure 8 — Scores comparatifs · Pressing · Création · Final

In [ ]:
CANDS_BAR = [('Thomasson\n(Réf.)', T, R)]
for kw, col in [('Longstaff',OR),('Gineitis',VT),('Diatta',PU),('Danois',B),('Ounahi',OG)]:
    r = MF[MF['player__name'].str.contains(kw, na=False, case=False)]
    if len(r):
        CANDS_BAR.append((r.iloc[0]['player__name'].split()[-1], r.iloc[0], col))

nn    = [n for n,_,_ in CANDS_BAR]
pp    = [r['s_press_n'] for _,r,_ in CANDS_BAR]
cc    = [r['s_crea_n']  for _,r,_ in CANDS_BAR]
ff    = [r['s_final_n'] for _,r,_ in CANDS_BAR]
cols_ = [c for _,_,c in CANDS_BAR]

fig, ax = plt.subplots(figsize=(14, 6), facecolor=F)
x, wd = np.arange(len(nn)), 0.28

ax.bar(x-wd, pp, wd, label='Pressing (45%)',
       color=[mcolors.to_rgba(c, 0.7) for c in cols_], edgecolor=F)
ax.bar(x,    cc, wd, label='Création (45%)',
       color=[mcolors.to_rgba(c, 0.5) for c in cols_], edgecolor=F)
ax.bar(x+wd, ff, wd, label='Score Final',
       color=cols_, edgecolor=W, lw=0.7)

for xi, fv, col in zip(x+wd, ff, cols_):
    ax.text(xi, fv+0.8, f'{fv:.0f}',
            ha='center', fontsize=9.5, color=col, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(nn, color=W, fontsize=10)
ax.set_ylim(0, 118)
ax.legend(fontsize=9, framealpha=0.3, labelcolor=W)
ax.set_title(
    'SCORES COMPARATIFS — Pressing · Création · Final\n'
    'TklW gagnés (FBRef) · xA · Centres précis · Imputation médiane',
    color=W, fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()

fname = f'{FIGDIR}/fig8_scores.png'
plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor=F)
plt.show()
print(f"✅ {fname}")


### Figure 9 — Podium final

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8), facecolor=F)
ax.set_facecolor(F); ax.axis('off')

def get_cand(kw):
    r = MF[MF['player__name'].str.contains(kw, na=False, case=False)]
    return r.iloc[0] if len(r) else None

ls, gi, di = get_cand('Longstaff'), get_cand('Gineitis'), get_cand('Diatta')
da, ou      = get_cand('Danois'),    get_cand('Ounahi')

PODIUM = [
    ('1', 'Sean Longstaff',  'Leeds United · PL',  '28 ans · ~8 M€',  ls, OR),
    ('2', 'Gvidas Gineitis', 'Torino · Serie A',   '21 ans · ~7 M€',  gi, VT),
    ('3', 'Krépin Diatta',   'AS Monaco · L1',     '26 ans · ~12 M€', di, PU),
]
HEIGHTS = [3.1, 2.5, 2.1]
STARTS  = [0.3, 4.8, 9.0]
WW = 4.0

for (rk, name, club, meta, r, col), h, sx in zip(PODIUM, HEIGHTS, STARTS):
    if r is None:
        continue
    sc    = r['s_final_n']
    stats = (f"{r.get('TklW_p90', 0):.2f} TklW/90\n"
             f"{r.get('interceptions_p90', 0):.2f} Int/90\n"
             f"{r.get('keyPasses_p90', 0):.2f} KP/90\n"
             f"{r.get('bigChancesCreated_p90', 0):.2f} BC/90\n"
             f"Rating {r.get('rating', 0):.2f}")

    ax.add_patch(plt.Rectangle((sx, 0.15), WW, h+0.4,
                               facecolor=col, alpha=0.10, edgecolor=col, lw=2))
    ax.text(sx+WW/2, h+0.26, f'#{rk}', ha='center', fontsize=32, color=col, fontweight='bold')
    ax.text(sx+WW/2, h-0.18, name,     ha='center', fontsize=13.5, color=W, fontweight='bold')
    ax.text(sx+WW/2, h-0.54, club,     ha='center', fontsize=9.5,  color=G)
    ax.text(sx+WW/2, h-0.86, meta,     ha='center', fontsize=9,    color=col, fontweight='bold')
    ax.text(sx+WW/2, h-1.42, f'{sc:.1f}/100',
            ha='center', fontsize=21, color=col, fontweight='bold')
    ax.text(sx+WW/2, 0.20, stats,
            ha='center', fontsize=8.5, color=G, va='bottom', linespacing=1.7)

# #4 et #5 sur la droite
if da is not None:
    ax.text(14.0, 3.2,
            f'#4  Kévin Danois\nAuxerre · L1\n21 ans · ~10 M€',
            ha='center', fontsize=10.5, color=B, fontweight='bold')
    ax.text(14.0, 2.1, f"{da['s_final_n']:.1f}/100",
            ha='center', fontsize=16, color=B, fontweight='bold')
    ax.text(14.0, 1.7, '⚠️ Score sous-estimé\n(contexte maintien Auxerre)',
            ha='center', fontsize=8, color=G, style='italic')
if ou is not None:
    ax.text(14.0, 0.95,
            f"#5  Azzedine Ounahi\nGirona · La Liga · 25 ans · ~10 M€",
            ha='center', fontsize=9.5, color=OG)
    ax.text(14.0, 0.50, f"{ou['s_final_n']:.1f}/100  ⚠️ Risque médical",
            ha='center', fontsize=9, color=OG)

ax.set_xlim(0, 16.2); ax.set_ylim(0, 4.4)
ax.set_title(
    'PODIUM FINAL — Thomasson Engine · RC Lens 2025/26\n'
    'Auteur : Zakaria · Pool vrais MF (positions FBRef filtrées) · RC Lens exclu · Imputation médiane · TklW FBRef',
    color=W, fontsize=12, fontweight='bold', y=0.99)
plt.tight_layout()

fname = f'{FIGDIR}/fig9_podium.png'
plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor=F)
plt.show()
print(f"✅ {fname}")


## 10. Conclusion & Recommandations

### Shortlist finale

| # | Joueur | Club | Âge | Budget | Décision |
|---|---|---|---|---|---|
| 🥇 1 | **Sean Longstaff** | Leeds United (PL) | 28 | ~8 M€ | **Priorité absolue** — profil hybride le plus complet dans le budget |
| 🥈 2 | **Gvidas Gineitis** | Torino (Serie A) | 21 | ~7 M€ | **Pépite Moneyball** — meilleur rapport valeur/prix, projet 3 ans |
| 🥉 3 | **Krépin Diatta** | AS Monaco (L1) | 26 | ~12 M€ | **Pressing élite** — stretch budget justifié par Int 1.69/90 |
| 4 | Kévin Danois | Auxerre (L1) | 21 | ~10 M€ | Score sous-estimé* (maintien) — potentiel si adaptation |
| ⚠️ 5 | Azzedine Ounahi | Girona (LaLiga) | 25 | ~10 M€ | Red flag médical — 3 blessures mollet cette saison |

### Cohérence budget RC Lens
- Historique : achats habituels **≤ 8-10 M€** · stretches exceptionnels (Openda 15 M€, Wahi 30 M€)
- Saison 25/26 : solde +45.6 M€ → **capacité à pousser jusqu'à 12-15 M€ si nécessaire**
- Longstaff (~8 M€) et Gineitis (~7 M€) sont dans la cible habituelle
- Diatta (~12 M€) est un stretch justifié par son niveau

### Limites de l'analyse (versions futures)
1. **Exporter FBRef Passing** → colonne `PrgP` (passes progressives) — métrique Box-to-Box absente ici
2. **Exporter FBRef Possession** → colonne `PrgC` (progressive carries) — progression balle au pied
3. **Normalisation P30 TIP/OTIP** (SkillCorner) → corrige le biais Danois/Auxerre et Longstaff/Championship
4. **On Ball Engagements** (SkillCorner, payant) → qualité réelle du pressing au-delà de TklW
5. **Distance Haute Intensité P90** (SkillCorner) → dimension physique Box-to-Box non couverte

### Profil Pierre Sage (Kan FC ep. 83)
> Le système de Sage exige un milieu **hybride** — capable de défendre haut, de récupérer le ballon **et** de le projeter rapidement vers l'avant. Thomasson est la clé de voûte de cette mécanique.  
> → Longstaff et Gineitis matchent exactement cette description.

In [ ]:
# ── Récapitulatif final ───────────────────────────────────────────────────────
print("=" * 72)
print("THOMASSON ENGINE — RÉCAPITULATIF FINAL")
print(f"Auteur     : Zakaria")
print(f"Pool       : {len(MF)} vrais milieux (positions FBRef filtrées) · RC Lens exclu")
print(f"Imputation : médiane par position (remplace 0 — robustesse statistique)")
print(f"Figures    : 9 fichiers dans {FIGDIR}/")
print("=" * 72)
print()

FINAL = [
    ('Longstaff', OR, '#1 ~8 M€   — Priorité absolue · Profil hybride le plus complet'),
    ('Gineitis',  VT, '#2 ~7 M€   — Pépite Moneyball · Meilleur V/P'),
    ('Diatta',    PU, '#3 ~12 M€  — Pressing élite · Stretch budget'),
    ('Danois',    B,  '#4 ~10 M€  — Sous-estimé (contexte maintien Auxerre)'),
    ('Ounahi',    OG, '#5 ~10 M€  — ⚠️ Risque médical (blessures mollet)'),
]
for kw, col, tag in FINAL:
    r = MF[MF['player__name'].str.contains(kw, na=False, case=False)]
    if len(r):
        r = r.iloc[0]
        print(f"[{tag}]")
        print(f"  {r['player__name']} · {r['team__name']} · {r.get('player_age', '?'):.0f} ans")
        print(f"  Score : {r['s_final_n']:.1f}/100   Press : {r['s_press_n']:.1f}   Crea : {r['s_crea_n']:.1f}")
        print(f"  TklW/90 : {r.get('TklW_p90', 0):.2f}  Int/90 : {r.get('interceptions_p90', 0):.2f}  "
              f"KP/90 : {r.get('keyPasses_p90', 0):.2f}  xA/90 : {r.get('expectedAssists_p90', 0):.3f}")
        print()

print("Figures générées :")
for f in sorted(os.listdir(FIGDIR)):
    if f.endswith('.png'):
        size = os.path.getsize(f'{FIGDIR}/{f}') // 1024
        print(f"  {FIGDIR}/{f}  ({size} Ko)")
